In [0]:
%run /Workspace/utilities

In [0]:
from datetime import date

In [0]:
df = spark.readStream.format('cloudFiles')\
    .option('cloudFiles.format','csv')\
    .option("cloudFiles.inferColumnTypes", "true")\
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")\
    .option('cloudFiles.schemaLocation','/Volumes/projectdev/bronze/plane_schema/')\
    .load(f'abfss://raw@datalakestrgdev.dfs.core.windows.net/PLANE/Datepart={date.today()}/')

In [0]:
display(
    df,
    checkpointLocation="/Volumes/projectdev/bronze/plane_schema/checkpoint1/"
)

In [0]:
dbutils.fs.rm('/Volumes/projectdev/bronze/plane_schema/checkpoint1/', True)

In [0]:
df_base = df.withColumnRenamed('tailnum','tailId')

df_base.writeStream.trigger(once=True)\
    .format('delta')\
    .option("mergeSchema", "true")\
    .outputMode('append')\
    .option('checkpointLocation','/Volumes/projectdev/bronze/plane_schema/checkpoint/')\
    .start('abfss://cleansed@datalakestrgdev.dfs.core.windows.net/plane')
     

In [0]:
df = spark.read.format('delta').load('abfss://cleansed@datalakestrgdev.dfs.core.windows.net/plane')

In [0]:
%sql
create table if not exists projectdev.cleansed.plane
using delta
location 'abfss://cleansed@datalakestrgdev.dfs.core.windows.net/plane'

In [0]:
%sql
select * from projectdev.cleansed.plane